# 13 — Polarimetric mode discovery and covariance inventory

**Goal:** find out which polarimetric analysis the available data allow, **before** choosing a
method.

* Quad-pol (HH, HV, VH, VV) allows the full 3x3 decompositions (Cloude–Pottier, Pauli,
  Freeman–Durden, Yamaguchi). Compact-pol (RH, RV) allows Stokes / m-χ.
* Over Europe NISAR acquires **dual-pol HH/HV** (`DHDH`) and some single-pol HH (`SHSH`).
  Quad-pol is planned only over specific areas (agriculture, cal/val), not on our sites.
* The dual-pol **GCOV** stores only `HHHH` and `HVHV`. The cross term ⟨S_HH S_HV*⟩ needed for
  dual-pol decompositions must be rebuilt from the complex **GSLC** (geocoded SLC) images.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nisar_s1.config import load_kit
from nisar_s1 import product as P

kit = load_kit()                                  # notebooks/kit_config.yaml
OUT = Path(kit["data_dir"]) / "outputs" / "13"   # outputs stay outside the repository
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from nisar_s1.config import load_params

## What NISAR acquired over the study area (ASF inventory of the feasibility study)

In [2]:
import nisar_s1.config as C
inv = pd.read_csv(C.project_root() / "results" / "tables" / "inventory_nisar_all.csv")
inv.groupby(["product", "pol_code"]).size().unstack(fill_value=0)

pol_code,DHDH,SH,SHSH
product,,,
GCOV,90,0,3
GOFF,0,51,0
GSLC,90,0,3
GUNW,0,67,0
RIFG,0,49,0
ROFF,0,51,0
RSLC,90,0,3
RUNW,0,49,0


## The mode, discovered from each product type

In [3]:
rows = []
with P.open_local_or_remote(kit["gcov"]["local"], kit["gcov"]["url"]) as h5:
    prof = P.product_profile(h5)
    for f in prof.grids:
        rows.append({"product": "GCOV", "frequency": f, "mode": prof.polarimetric_mode,
                     "layers": prof.grids[f].terms, "complex cross term": prof.complex_terms_available(f)})
gdir = Path(kit["data_dir"]) / "gslc"
gslc_files = sorted(gdir.glob("*.h5"))
for p in gslc_files[:1]:
    import h5py
    with h5py.File(p) as h5:
        sp = P.product_profile(h5)
        g = sp.grids["frequencyA"]
        rows.append({"product": "GSLC (AOI subset)", "frequency": "frequencyA", "mode": sp.polarimetric_mode,
                     "layers": g.polarizations, "complex cross term": str(h5[f"{g.path}/HH"].dtype).startswith("complex")})
inventory = pd.DataFrame(rows); inventory

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5


,product,frequency,mode,layers,complex cross term
0,GCOV,frequencyA,dual-pol linear (HH/HV),"[HHHH, HVHV]",False
1,GCOV,frequencyB,dual-pol linear (HH/HV),"[HHHH, HVHV]",False
2,GSLC (AOI subset),frequencyA,dual-pol linear (HH/HV),"[HH, HV]",True


### Decision table: which methods of the original (quad/compact-pol) toolkit carry over?

| Original module | Needs | With NISAR dual-pol HH/HV | This series |
|---|---|---|---|
| 14 full-pol C3 reconstruction | 9 terms | 2x2 C2 from GSLC | **14** C2 from GSLC + validation against GCOV |
| 15 eigen-analysis | C3 | eigen-analysis of C2 | **15** |
| 16 Cloude–Pottier H/A/α | C3 | dual-pol H/α (Cloude 2007) | **16** |
| 17 Pauli, Freeman–Durden, Yamaguchi | VV and HH–VV phase | not possible | **17** dual-pol indices (DpRVI, RVI, Stokes) |
| 18 compact-pol (S-band SSAR) | RH/RV | no S-band over Europe | **18** Sentinel-1 C-band C2 (VV/VH), same methods |
| 19 capstone | — | — | **19** clear-cut seen by L- and C-band polarimetry |

The GSLC subsets used in 14–19 are prepared once by `scripts/kit/k1_prepare_data.py`: only the AOI
window of each 13 GB GSLC is streamed from ASF and saved with the same HDF5 layout.

In [4]:
P.save_session(kit["session_file"], gslc_subsets=[str(p) for p in gslc_files])
print(len(gslc_files), "GSLC subsets:", [p.name[:60] for p in gslc_files])

2 GSLC subsets: ['NISAR_L2_PR_GSLC_025_144_A_026_2005_DHDH_M_20260718T041939_2', 'NISAR_L2_PR_GSLC_028_144_A_026_2005_DHDH_M_20260823T041937_2']


In [5]:
checks = {
    "GCOV has no complex cross term": not inventory.loc[0, 'complex cross term'],
    "GSLC subsets present": len(gslc_files) >= 2,
    "GSLC is complex dual-pol": bool(inventory.iloc[-1]['complex cross term']),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
assert all(checks.values()), "some checks failed: read the messages above"
print("Module 13 STATUS: PASS")

  [x] GCOV has no complex cross term
  [x] GSLC subsets present
  [x] GSLC is complex dual-pol
Module 13 STATUS: PASS
